# 02 — Mark 2 — Predicted-Liver ROI & Multi-Window Feasibility

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **02 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_2_roi_multiwindow_feasibility.ipynb`

## Objective

Does cropping the image to the predicted-liver ROI (instead of the full 256×256 slice) contain all tumour pixels, and do multi-window HU representations (broad, liver, robust-normalized) preserve that signal?

## Inputs (read-only)

- `mark_1_gate_result.json` (reads from shared `mark_1_to_4e_outputs/mark_1_outputs/`)
- Frozen Mark 1 probability cache (REUSE) or live inference (REBUILD)

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_2_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_2_gate_result.json` |
| `cache_coverage.csv` |
| `roi_configuration_results.csv` |
| `roi_patient_results.csv` |
| `multiwindow_summary.csv` |
| `multiwindow_slice_statistics.csv` |
| `v104_v116_window_summary.csv` |
| `roi_feasibility_dashboard.png` |
| `selected_roi_v104_v116.png` |
| `v104_v116_multiwindow_examples.png` |
| `multiwindow_feasibility_dashboard.png` |

**Visualizations produced by this notebook:** `roi_feasibility_dashboard.png`, `selected_roi_v104_v116.png`, `v104_v116_multiwindow_examples.png`, `multiwindow_feasibility_dashboard.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: mark_1_gate_result.json, Frozen Mark 1 probability cache (REUSE) or live inference (REBUILD)"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_2_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**ROI feasibility PASSED for containment**: the predicted-liver ROI contains ~100% of tumour pixels while cropping to a median ~42.7% of the image — the two-stage (ROI → tumour) plan is viable.

## Gate

`mark_2_gate_result.json` — ROI feasibility gate (tumour containment / crop budget)

## Run notes

No training, no patient-specific ROI, no test access. Output crops are deterministic; the selected ROI configuration feeds Mark 3.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_2"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 2 — Mark 2: Predicted-Liver ROI & Multi-Window Feasibility

**Original:** `mark 1/mark_2_roi_multiwindow_feasibility.ipynb`

## Question

Before another tumor model is trained: can one **global, prediction-only** 3D liver ROI contain every
validation tumor (including V104/V116), and do fixed source-HU windows improve tumor/liver separation?

## Key finding (reproduced)

- **ROI gate: PASSED.** liver threshold 0.50, padding 16, `largest_3d` component:
  V104 and V116 tumor containment = 1.0, minimum positive-patient/slice containment = 1.0,
  zero empty patient ROIs, **median crop-area ratio 0.427** (~58% slice-search reduction).
- Fixed windows `broad [-160,240]`, `liver [-20,140]`, `narrow [20,120]` are diagnostic inputs
  for Mark 3, not yet proven model improvements.

## Contract

- ROI uses **predicted-liver probabilities only** — ground truth is never an ROI input.
- Feasibility gate: V104 ≥ 99%, V116 ≥ 99%, min positive-patient containment ≥ 99%,
  min positive-slice containment ≥ 99%, zero empty ROIs, median crop ratio ≤ 60%.
- Test split locked.

### 2.1 Load the Mark 1 caches and verify the Mark 1 gate

In [2]:
import shutil

mark1_gate = require_upstream_gate("mark_1")
assert mark1_gate["status"] == "mark_1_diagnostic_complete"
assert mark1_gate["calibration_gate_passed"] is False
assert mark1_gate["test_images_accessed"] is False

m2_cache = {}
for path in sorted((OUT_CACHE["mark_1"]).glob("volume_*.npz")):
    volume_id = int(path.stem.split("_")[-1])
    with np.load(path, allow_pickle=False) as payload:
        m2_cache[volume_id] = {key: payload[key] for key in payload.files}
assert len(m2_cache) == 13
cache_coverage = pd.DataFrame([
    {"volume_id": v, "slices": len(i["sample_id"]),
     "tumor_pixels": int(i["tumor_truth"].sum()),
     "tumor_positive_slices": int(i["tumor_truth"].any(axis=(1, 2)).sum())}
    for v, i in m2_cache.items()])
cache_coverage.to_csv(OUT["mark_2"] / "cache_coverage.csv", index=False)
display(cache_coverage)
print("PASS: 13 complete, ordered probability caches reused from Mark 1.")

,volume_id,slices,tumor_pixels,tumor_positive_slices
0,104,781,69032,122
1,105,986,0,0
2,106,771,0,0
3,107,771,1473,71
4,108,856,363698,202
5,109,756,18200,131
6,110,816,31120,130
7,111,761,1830,92
8,112,751,295,26
9,113,836,21076,135


PASS: 13 complete, ordered probability caches reused from Mark 1.


### 2.2 Construct prediction-only 3D ROIs and score containment

In [3]:
from scipy import ndimage

ROI_LIVER_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50]
PADDINGS = [16, 32, 48, 64]
COMPONENT_MODES = ["all", "largest_3d"]


def predicted_liver_mask(probability, threshold, component_mode):
    mask = probability.astype(np.float32) >= threshold
    if component_mode == "all" or not mask.any():
        return mask
    labels, count = ndimage.label(mask, structure=np.ones((3, 3, 3), dtype=np.uint8))
    if count == 0:
        return mask
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == sizes.argmax()


def bbox_from_mask(mask, padding=0):
    if not mask.any():
        return None
    _, ys, xs = np.where(mask)
    return (max(int(ys.min()) - padding, 0), min(int(ys.max()) + 1 + padding, 256),
            max(int(xs.min()) - padding, 0), min(int(xs.max()) + 1 + padding, 256))


def padded_box(base_box, padding):
    if base_box is None:
        return None
    y0, y1, x0, x1 = base_box
    return (max(y0 - padding, 0), min(y1 + padding, 256),
            max(x0 - padding, 0), min(x1 + padding, 256))


def score_roi(volume_id, item, threshold, padding, component_mode, base_box):
    box = padded_box(base_box, padding)
    truth = item["tumor_truth"].astype(bool)
    positive_slices = truth.any(axis=(1, 2))
    if box is None:
        contained = np.zeros_like(truth)
        area_ratio = 0.0
    else:
        y0, y1, x0, x1 = box
        contained = np.zeros_like(truth)
        contained[:, y0:y1, x0:x1] = truth[:, y0:y1, x0:x1]
        area_ratio = ((y1 - y0) * (x1 - x0)) / (256 * 256)
    contained_pixels = int(contained.sum())
    truth_pixels = int(truth.sum())
    contained_positive_slices = (contained.any(axis=(1, 2)) & positive_slices).sum()
    return {
        "volume_id": volume_id, "liver_threshold": threshold, "padding": padding,
        "component_mode": component_mode, "roi_empty": box is None,
        "y0": box[0] if box else np.nan, "y1": box[1] if box else np.nan,
        "x0": box[2] if box else np.nan, "x1": box[3] if box else np.nan,
        "crop_area_ratio": area_ratio, "tumor_pixels": truth_pixels,
        "tumor_pixel_containment": (contained_pixels / truth_pixels if truth_pixels else np.nan),
        "positive_slices": int(positive_slices.sum()),
        "positive_slice_containment": (contained_positive_slices / positive_slices.sum()
                                       if positive_slices.any() else np.nan),
    }


roi_patient_rows = []
for threshold in ROI_LIVER_THRESHOLDS:
    for component_mode in COMPONENT_MODES:
        for volume_id, item in m2_cache.items():
            liver_mask = predicted_liver_mask(item["liver_probability"], threshold, component_mode)
            base_box = bbox_from_mask(liver_mask)
            for padding in PADDINGS:
                roi_patient_rows.append(score_roi(volume_id, item, threshold, padding,
                                                  component_mode, base_box))
roi_patient_results = pd.DataFrame(roi_patient_rows)
roi_patient_results.to_csv(OUT["mark_2"] / "roi_patient_results.csv", index=False)
print(f"Evaluated {len(roi_patient_results):,} patient/configuration rows.")

Evaluated 520 patient/configuration rows.


### 2.3 Aggregate configurations and apply the feasibility gate

In [4]:
configuration_rows = []
group_columns = ["liver_threshold", "padding", "component_mode"]
for keys, group in roi_patient_results.groupby(group_columns, sort=True):
    threshold, padding, component_mode = keys
    positive = group.loc[group["tumor_pixels"].gt(0)]
    indexed = group.set_index("volume_id")
    configuration_rows.append({
        "liver_threshold": threshold, "padding": padding,
        "component_mode": component_mode,
        "volume_104_tumor_containment": indexed.at[104, "tumor_pixel_containment"],
        "volume_116_tumor_containment": indexed.at[116, "tumor_pixel_containment"],
        "minimum_positive_patient_containment": positive["tumor_pixel_containment"].min(),
        "mean_positive_patient_containment": positive["tumor_pixel_containment"].mean(),
        "minimum_positive_slice_containment": positive["positive_slice_containment"].min(),
        "median_crop_area_ratio": group["crop_area_ratio"].median(),
        "maximum_crop_area_ratio": group["crop_area_ratio"].max(),
        "empty_patient_rois": int(group["roi_empty"].sum()),
    })

roi_configurations = pd.DataFrame(configuration_rows)
roi_configurations["hard_containment_gate_passed"] = (
    roi_configurations["volume_104_tumor_containment"].ge(0.99)
    & roi_configurations["volume_116_tumor_containment"].ge(0.99)
    & roi_configurations["minimum_positive_patient_containment"].ge(0.99)
    & roi_configurations["minimum_positive_slice_containment"].ge(0.99)
    & roi_configurations["empty_patient_rois"].eq(0))
roi_configurations["efficient_roi_gate_passed"] = (
    roi_configurations["hard_containment_gate_passed"]
    & roi_configurations["median_crop_area_ratio"].le(0.60))
roi_configurations.to_csv(OUT["mark_2"] / "roi_configuration_results.csv", index=False)

eligible = roi_configurations.loc[roi_configurations["efficient_roi_gate_passed"]].copy()
fallback = roi_configurations.loc[roi_configurations["hard_containment_gate_passed"]].copy()
candidate_pool = eligible if not eligible.empty else fallback
if not candidate_pool.empty:
    selected_roi = candidate_pool.sort_values(
        ["median_crop_area_ratio", "minimum_positive_patient_containment"],
        ascending=[True, False]).iloc[0]
else:
    selected_roi = roi_configurations.sort_values(
        ["minimum_positive_patient_containment", "minimum_positive_slice_containment",
         "median_crop_area_ratio"], ascending=[False, False, True]).iloc[0]

display(roi_configurations.sort_values(
    ["hard_containment_gate_passed", "minimum_positive_patient_containment",
     "median_crop_area_ratio"], ascending=[False, False, True]).head(20))
print("Selected diagnostic ROI:", selected_roi.to_dict())

,liver_threshold,padding,component_mode,volume_104_tumor_containment,volume_116_tumor_containment,minimum_positive_patient_containment,mean_positive_patient_containment,minimum_positive_slice_containment,median_crop_area_ratio,maximum_crop_area_ratio,empty_patient_rois,hard_containment_gate_passed,efficient_roi_gate_passed
33,0.5,16,largest_3d,1.0,1.0,1.0,1.0,1.0,0.427002,0.533203,0,True,True
25,0.4,16,largest_3d,1.0,1.0,1.0,1.0,1.0,0.429688,0.533203,0,True,True
17,0.3,16,largest_3d,1.0,1.0,1.0,1.0,1.0,0.433594,0.533203,0,True,True
9,0.2,16,largest_3d,1.0,1.0,1.0,1.0,1.0,0.437286,0.535980,0,True,True
1,0.1,16,largest_3d,1.0,1.0,1.0,1.0,1.0,0.447113,0.541718,0,True,True
32,0.5,16,all,1.0,1.0,1.0,1.0,1.0,0.580078,0.726318,0,True,True
24,0.4,16,all,1.0,1.0,1.0,1.0,1.0,0.581863,0.772705,0,True,True
35,0.5,32,largest_3d,1.0,1.0,1.0,1.0,1.0,0.594727,0.721649,0,True,True
27,0.4,32,largest_3d,1.0,1.0,1.0,1.0,1.0,0.598145,0.721649,0,True,True
19,0.3,32,largest_3d,1.0,1.0,1.0,1.0,1.0,0.607773,0.721649,0,True,False


Selected diagnostic ROI: {'liver_threshold': 0.5, 'padding': 16, 'component_mode': 'largest_3d', 'volume_104_tumor_containment': 1.0, 'volume_116_tumor_containment': 1.0, 'minimum_positive_patient_containment': 1.0, 'mean_positive_patient_containment': 1.0, 'minimum_positive_slice_containment': 1.0, 'median_crop_area_ratio': 0.427001953125, 'maximum_crop_area_ratio': 0.533203125, 'empty_patient_rois': 0, 'hard_containment_gate_passed': True, 'efficient_roi_gate_passed': True}


### 2.4 ROI feasibility dashboards

In [5]:
figure, axes = plt.subplots(2, 2, figsize=(17, 12))
for mode, marker in [("all", "o"), ("largest_3d", "s")]:
    subset = roi_configurations.loc[roi_configurations["component_mode"].eq(mode)]
    axes[0, 0].scatter(subset["median_crop_area_ratio"],
                       subset["minimum_positive_patient_containment"],
                       label=mode, marker=marker, s=60, alpha=0.8)
axes[0, 0].axhline(0.99, linestyle="--", color="#444444")
axes[0, 0].axvline(0.60, linestyle=":", color="#444444")
axes[0, 0].set_xlabel("Median crop-area ratio")
axes[0, 0].set_ylabel("Minimum positive-patient containment")
axes[0, 0].set_title("Containment versus crop burden"); axes[0, 0].legend()

pivot_104 = roi_configurations.loc[
    roi_configurations["component_mode"].eq("all")].pivot(
    index="liver_threshold", columns="padding", values="volume_104_tumor_containment")
image = axes[0, 1].imshow(pivot_104, vmin=0, vmax=1, cmap="viridis", aspect="auto")
axes[0, 1].set_xticks(range(len(pivot_104.columns)), pivot_104.columns)
axes[0, 1].set_yticks(range(len(pivot_104.index)), pivot_104.index)
axes[0, 1].set_xlabel("Padding"); axes[0, 1].set_ylabel("Liver threshold")
axes[0, 1].set_title("V104 containment — all components")
figure.colorbar(image, ax=axes[0, 1], fraction=0.046)

pivot_116 = roi_configurations.loc[
    roi_configurations["component_mode"].eq("all")].pivot(
    index="liver_threshold", columns="padding", values="volume_116_tumor_containment")
image = axes[1, 0].imshow(pivot_116, vmin=0, vmax=1, cmap="viridis", aspect="auto")
axes[1, 0].set_xticks(range(len(pivot_116.columns)), pivot_116.columns)
axes[1, 0].set_yticks(range(len(pivot_116.index)), pivot_116.index)
axes[1, 0].set_xlabel("Padding"); axes[1, 0].set_ylabel("Liver threshold")
axes[1, 0].set_title("V116 containment — all components")
figure.colorbar(image, ax=axes[1, 0], fraction=0.046)

selected_filter = (
    roi_patient_results["liver_threshold"].eq(selected_roi["liver_threshold"])
    & roi_patient_results["padding"].eq(selected_roi["padding"])
    & roi_patient_results["component_mode"].eq(selected_roi["component_mode"]))
selected_patients = roi_patient_results.loc[selected_filter].sort_values("volume_id")
axes[1, 1].bar(selected_patients["volume_id"].astype(str),
               selected_patients["tumor_pixel_containment"].fillna(1.0),
               color=["#2878B5" if v >= 0.99 else "#F28E2B"
                      for v in selected_patients["tumor_pixel_containment"].fillna(1.0)])
axes[1, 1].axhline(0.99, linestyle="--", color="#444444")
axes[1, 1].set_ylim(0, 1.03)
axes[1, 1].set_title("Selected ROI tumor containment by patient")
axes[1, 1].set_xlabel("Volume"); axes[1, 1].set_ylabel("Containment")

figure.suptitle("Predicted-liver ROI feasibility", fontsize=18, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_2"] / "roi_feasibility_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_12220\4094284619.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.5 Inspect selected predicted ROIs for V104 and V116

In [6]:
def load_derived_image(sample_id):
    row = validation_manifest.loc[validation_manifest["sample_id"].eq(sample_id)].iloc[0]
    with Image.open(DATASET_ROOT / row["image_path"]) as handle:
        return np.asarray(handle.convert("L"), dtype=np.float32) / 255.0


def selected_box_for_volume(volume_id):
    row = roi_patient_results.loc[
        selected_filter & roi_patient_results["volume_id"].eq(volume_id)].iloc[0]
    if row["roi_empty"]:
        return None
    return tuple(int(row[key]) for key in ("y0", "y1", "x0", "x1"))


figure, axes = plt.subplots(2, 4, figsize=(17, 9))
for row_axes, volume_id in zip(axes, [104, 116]):
    item = m2_cache[volume_id]
    tumor_sizes = item["tumor_truth"].sum(axis=(1, 2))
    index = int(np.argmax(tumor_sizes))
    sample_id = str(item["sample_id"][index])
    image = load_derived_image(sample_id)
    liver_probability = item["liver_probability"][index].astype(np.float32)
    tumor = item["tumor_truth"][index].astype(bool)
    box = selected_box_for_volume(volume_id)

    row_axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
    row_axes[0].contour(tumor, levels=[0.5], colors=["#00FFFF"])
    row_axes[0].set_title(f"V{volume_id} largest tumor slice")
    row_axes[1].imshow(liver_probability, cmap="viridis", vmin=0, vmax=1)
    row_axes[1].set_title("Predicted-liver probability")
    row_axes[2].imshow(image, cmap="gray", vmin=0, vmax=1)
    row_axes[2].contour(tumor, levels=[0.5], colors=["#00FFFF"])
    if box:
        y0, y1, x0, x1 = box
        row_axes[2].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                            fill=False, edgecolor="#FF2D2D", linewidth=2))
    row_axes[2].set_title("Prediction-only 3D ROI")
    if box:
        y0, y1, x0, x1 = box
        row_axes[3].imshow(image[y0:y1, x0:x1], cmap="gray", vmin=0, vmax=1)
    else:
        row_axes[3].text(0.5, 0.5, "EMPTY ROI", ha="center", va="center")
    row_axes[3].set_title("ROI crop")
    for axis in row_axes:
        axis.axis("off")
figure.suptitle("Selected ROI inspection", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_2"] / "selected_roi_v104_v116.png", dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_12220\2968638454.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.6 Fixed-window source-HU visibility (train + validation)

In [7]:
import nibabel as nib

WINDOWS = {"broad_abdominal": (-160.0, 240.0), "liver_soft_tissue": (-20.0, 140.0),
           "narrow_lesion": (20.0, 120.0)}


def transform_labels(labels, transform):
    if transform == "identity":
        return labels
    if transform == "rot180":
        return np.rot90(labels, 2).copy()
    raise ValueError(transform)


def window_statistics(split):
    rows = []
    split_manifest = manifest.loc[
        manifest["split"].eq(split) & manifest["tumor_pixels"].gt(0)]
    for volume_id, group in split_manifest.groupby("volume_id", sort=True):
        first = group.iloc[0]
        ct = nib.load(str(first["source_volume_path"]))
        segmentation = nib.load(str(first["source_segmentation_path"]))
        transform = str(first["transform_applied"])
        for row in group.sort_values("slice_index").itertuples(index=False):
            z = int(row.slice_index)
            hu = np.asanyarray(ct.dataobj[:, :, z]).astype(np.float32)
            labels = transform_labels(
                np.asanyarray(segmentation.dataobj[:, :, z]).astype(np.uint8), transform)
            tumor = hu[labels == 2]
            liver = hu[labels == 1]
            if not tumor.size or not liver.size:
                continue
            base = {"split": split, "sample_id": row.sample_id, "volume_id": int(volume_id),
                    "slice_index": z, "tumor_pixels_native": int(tumor.size),
                    "median_contrast_hu": float(np.median(tumor) - np.median(liver))}
            for name, (lower, upper) in WINDOWS.items():
                tumor_window = window_hu(tumor, (lower, upper))
                liver_window = window_hu(liver, (lower, upper))
                rows.append({
                    **base, "window": name, "lower_hu": lower, "upper_hu": upper,
                    "tumor_median_windowed": float(np.median(tumor_window)),
                    "liver_median_windowed": float(np.median(liver_window)),
                    "absolute_median_separation": float(
                        abs(np.median(tumor_window) - np.median(liver_window))),
                    "tumor_low_saturation_pct": float(100 * (tumor <= lower).mean()),
                    "tumor_high_saturation_pct": float(100 * (tumor >= upper).mean()),
                    "liver_low_saturation_pct": float(100 * (liver <= lower).mean()),
                    "liver_high_saturation_pct": float(100 * (liver >= upper).mean()),
                })
    return pd.DataFrame(rows)


window_slice_statistics = pd.concat(
    [window_statistics("train"), window_statistics("val")], ignore_index=True)
window_slice_statistics.to_csv(OUT["mark_2"] / "multiwindow_slice_statistics.csv", index=False)
window_summary = window_slice_statistics.groupby(["split", "window"]).agg(
    slices=("sample_id", "size"),
    median_absolute_separation=("absolute_median_separation", "median"),
    median_tumor_low_saturation_pct=("tumor_low_saturation_pct", "median"),
    median_tumor_high_saturation_pct=("tumor_high_saturation_pct", "median")).reset_index()
window_summary.to_csv(OUT["mark_2"] / "multiwindow_summary.csv", index=False)
display(window_summary)

,split,window,slices,median_absolute_separation,median_tumor_low_saturation_pct,median_tumor_high_saturation_pct
0,train,broad_abdominal,4923,0.09750,0.000000,0.000000
1,train,liver_soft_tissue,4923,0.24375,0.138122,0.330868
2,train,narrow_lesion,4923,0.38000,7.638889,2.795031
3,val,broad_abdominal,1042,0.10250,0.000000,0.000000
4,val,liver_soft_tissue,1042,0.25625,0.000000,0.294155
5,val,narrow_lesion,1042,0.38000,5.250704,2.355586


### 2.7 Window separability dashboards + V104/V116 examples

In [8]:
figure, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for name in WINDOWS:
    train_values = window_slice_statistics.loc[
        window_slice_statistics["split"].eq("train") & window_slice_statistics["window"].eq(name),
        "absolute_median_separation"]
    val_values = window_slice_statistics.loc[
        window_slice_statistics["split"].eq("val") & window_slice_statistics["window"].eq(name),
        "absolute_median_separation"]
    axes[0].hist(train_values, bins=40, density=True, histtype="step",
                 linewidth=1.5, label=f"{name} train")
    axes[0].hist(val_values, bins=40, density=True, histtype="step",
                 linewidth=1.5, linestyle="--", label=f"{name} val")
axes[0].set_title("Tumor–liver windowed separation")
axes[0].set_xlabel("Absolute median separation"); axes[0].set_ylabel("Density")
axes[0].legend(fontsize=7)

focus = window_slice_statistics.loc[
    window_slice_statistics["split"].eq("val")
    & window_slice_statistics["volume_id"].isin([104, 116])]
focus_box = [focus.loc[focus["volume_id"].eq(v) & focus["window"].eq(name),
                       "absolute_median_separation"].to_numpy()
             for v in (104, 116) for name in WINDOWS]
labels = [f"V{v}\n{name}" for v in (104, 116) for name in WINDOWS]
axes[1].boxplot(focus_box, tick_labels=labels, showfliers=False)
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("V104/V116 separation by window")
axes[1].set_ylabel("Absolute median separation")

summary_pivot = window_summary.pivot(index="window", columns="split",
                                     values="median_absolute_separation")
summary_pivot.plot.bar(ax=axes[2], color=["#2878B5", "#F28E2B"])
axes[2].set_title("Median separation by split")
axes[2].set_ylabel("Absolute median separation")
axes[2].tick_params(axis="x", rotation=25)
axes[2].legend(title="Split")

figure.suptitle("Fixed source-HU window feasibility", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_2"] / "multiwindow_feasibility_dashboard.png",
               dpi=170, bbox_inches="tight")
plt.show()


def source_slice(volume_id, slice_index):
    row = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)
        & validation_manifest["slice_index"].eq(slice_index)].iloc[0]
    ct = nib.load(str(row["source_volume_path"]))
    segmentation = nib.load(str(row["source_segmentation_path"]))
    hu = np.asanyarray(ct.dataobj[:, :, slice_index]).astype(np.float32)
    labels = transform_labels(
        np.asanyarray(segmentation.dataobj[:, :, slice_index]).astype(np.uint8),
        str(row["transform_applied"]))
    return hu, labels


figure, axes = plt.subplots(2, len(WINDOWS) + 1, figsize=(18, 9))
for row_axes, volume_id in zip(axes, [104, 116]):
    item = m2_cache[volume_id]
    index = int(np.argmax(item["tumor_truth"].sum(axis=(1, 2))))
    slice_index = int(item["slice_index"][index])
    hu, labels = source_slice(volume_id, slice_index)
    tumor = labels == 2
    row_axes[0].imshow(hu, cmap="gray", vmin=-160, vmax=240)
    row_axes[0].contour(tumor, levels=[0.5], colors=["#00FFFF"])
    row_axes[0].set_title(f"V{volume_id} broad + truth")
    for axis, (name, (lower, upper)) in zip(row_axes[1:], WINDOWS.items()):
        axis.imshow(window_hu(hu, (lower, upper)), cmap="gray", vmin=0, vmax=1)
        axis.contour(tumor, levels=[0.5], colors=["#00FFFF"])
        axis.set_title(f"{name}\n[{lower:.0f}, {upper:.0f}] HU")
    for axis in row_axes:
        axis.axis("off")
figure.suptitle("V104 and V116 fixed-window inspection", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_2"] / "v104_v116_multiwindow_examples.png",
               dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_12220\3098476248.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_12220\3098476248.py:77: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.8 Write the Mark 2 feasibility gate

In [9]:
roi_hard_passed = bool(selected_roi["hard_containment_gate_passed"])
roi_efficient_passed = bool(selected_roi["efficient_roi_gate_passed"])

focus_window_summary = window_slice_statistics.loc[
    window_slice_statistics["split"].eq("val")
    & window_slice_statistics["volume_id"].isin([104, 116])].groupby(
    ["volume_id", "window"]).agg(
    median_separation=("absolute_median_separation", "median"),
    tumor_low_saturation_pct=("tumor_low_saturation_pct", "median"),
    tumor_high_saturation_pct=("tumor_high_saturation_pct", "median")).reset_index()
focus_window_summary.to_csv(OUT["mark_2"] / "v104_v116_window_summary.csv", index=False)

if not roi_hard_passed:
    decision, next_notebook = "REVISE_LIVER_LOCALIZATION_OR_USE_SAFER_ANATOMY_ROI", "mark_2b_liver_roi_recovery"
elif roi_hard_passed and not roi_efficient_passed:
    decision, next_notebook = "ROI_CONTAINS_TUMOR_BUT_CROP_IS_TOO_BROAD", "mark_2b_roi_efficiency_ablation"
else:
    decision, next_notebook = "PROCEED_TO_GATED_TWO_STAGE_MULTIWINDOW_OVERFIT", "mark_3_two_stage_multiwindow_overfit"

m2_gate = {
    "status": "mark_2_feasibility_complete",
    "roi_hard_containment_gate_passed": roi_hard_passed,
    "roi_efficiency_gate_passed": roi_efficient_passed,
    "selected_roi_configuration": {
        key: (selected_roi[key].item() if hasattr(selected_roi[key], "item") else selected_roi[key])
        for key in selected_roi.index},
    "fixed_windows": {key: list(value) for key, value in WINDOWS.items()},
    "decision": decision, "next_notebook": next_notebook,
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "mark1_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
    "roi_uses_ground_truth": False, "test_images_accessed": False,
}
(OUT["mark_2"] / "mark_2_gate_result.json").write_text(json.dumps(m2_gate, indent=2))
display(pd.DataFrame([m2_gate]).T.rename(columns={0: "value"}))
display(focus_window_summary)
print("DECISION:", decision)

# ---- Reproduction check against the original gate ----
orig_m2 = json.loads((MARK1_DIR / "mark_2_outputs" / "mark_2_gate_result.json").read_text())
orig_sel = orig_m2["selected_roi_configuration"]
recomputed_sel = m2_gate["selected_roi_configuration"]
key_set = ["volume_104_tumor_containment", "volume_116_tumor_containment",
           "minimum_positive_patient_containment", "minimum_positive_slice_containment",
           "median_crop_area_ratio", "maximum_crop_area_ratio", "empty_patient_rois"]
diffs = {k: abs(float(recomputed_sel[k]) - float(orig_sel[k])) for k in key_set}
print("Mark 2 reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 2 ROI gate drifted from the original!"
assert m2_gate["decision"] == orig_m2["decision"]
print("PASS: Mark 2 gate matches the original mark_2_gate_result.json.")

,value
status,mark_2_feasibility_complete
roi_hard_containment_gate_passed,True
roi_efficiency_gate_passed,True
selected_roi_configuration,"{'liver_threshold': 0.5, 'padding': 16, 'compo..."
fixed_windows,"{'broad_abdominal': [-160.0, 240.0], 'liver_so..."
decision,PROCEED_TO_GATED_TWO_STAGE_MULTIWINDOW_OVERFIT
next_notebook,mark_3_two_stage_multiwindow_overfit
manifest_sha256,575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63...
mark1_checkpoint_sha256,9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c...
roi_uses_ground_truth,False


,volume_id,window,median_separation,tumor_low_saturation_pct,tumor_high_saturation_pct
0,104,broad_abdominal,0.223750,0.000000,0.000000
1,104,liver_soft_tissue,0.559375,5.944323,0.000000
2,104,narrow_lesion,0.780000,71.602424,0.000000
3,116,broad_abdominal,0.025000,0.000000,0.000000
4,116,liver_soft_tissue,0.062500,0.611621,0.903614
5,116,narrow_lesion,0.100000,11.697008,2.541528


DECISION: PROCEED_TO_GATED_TWO_STAGE_MULTIWINDOW_OVERFIT
Mark 2 reproduction check: {'volume_104_tumor_containment': 0.0, 'volume_116_tumor_containment': 0.0, 'minimum_positive_patient_containment': 0.0, 'minimum_positive_slice_containment': 0.0, 'median_crop_area_ratio': 0.0, 'maximum_crop_area_ratio': 0.0, 'empty_patient_rois': 0.0}
PASS: Mark 2 gate matches the original mark_2_gate_result.json.


In [10]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_2")


PASS: mark_2_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\02_mark_2\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_12220\2180361621.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ---------------------------------------------------------------------------
# Publish key mark_2 artifacts to the shared/legacy folder other code reads.
#
# External consumers: original `mark 1/` mark_3 & mark_4 notebooks (`mark_2_gate_result.json`, `roi_patient_results.csv`).
# These code files hardcode artifact paths under `mark 1/mark_2_outputs/`, so
# after every run the freshly produced artifacts are mirrored there to keep
# those code files working. Values are recomputed from frozen inputs and
# verified against the original gates (reproduction check above), so the
# mirrored files are equivalent.
# ---------------------------------------------------------------------------
import shutil

PUBLISH_DIR = MARK1_DIR / "mark_2_outputs"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

published = []
for pattern in ("*.csv", "*.json", "*.pth"):
    for source in sorted(OUT["mark_2"].glob(pattern)):
        shutil.copy2(source, PUBLISH_DIR / source.name)
        published.append(PUBLISH_DIR / source.name)

assert published, f"no mark_2 artifacts found to publish"
print(f"PUBLISHED {len(published)} mark_2 artifacts to {PUBLISH_DIR}:")
for artifact in sorted(published):
    print("  " + str(artifact))

PUBLISHED 7 mark_2 artifacts to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs\cache_coverage.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs\mark_2_gate_result.json
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs\multiwindow_slice_statistics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs\multiwindow_summary.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs\roi_configuration_results.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs\roi_patient_results.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_2_outputs\v104_v116_window_summary.csv
